# M1 · First Inference

> **Goal:** make your first model calls on Foundry — chat, embeddings, streaming, and the Responses API — all from one client.
> **You'll use:** `AIProjectClient`, `get_openai_client()`, `chat.completions`, `embeddings`, `responses`.

---

Every lab in this workshop starts the same way: authenticate with
`DefaultAzureCredential`, build an **`AIProjectClient`** from your project endpoint,
and ask it for an OpenAI-compatible client. That one client gives you **chat**,
**embeddings**, and the **Responses API** that later powers agents and tools.

![The inference path](../../assets/inference-path.png)

If you haven't set up your project and `.env` yet, do the
[Setup](../../setup/) first.

In [1]:
# print current date and time
from datetime import datetime

# Get the current date and time
current_datetime = datetime.now()

# Print the current date and time
print("Current date and time:", current_datetime)

Current date and time: 2026-08-27 10:00:53.476402


## 1. Configure

Every lab reads the same variables from your `.env` (see
[Setup](../../setup/)). We load them and grab the two model
deployment names we'll use here.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # reads .env from the repo root

# only print first 15 characters of project endpoint for security reasons
PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
CHAT_MODEL       = os.environ.get("CHAT_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL  = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-large")

print("Project :", PROJECT_ENDPOINT[:15])
print("Chat    :", CHAT_MODEL)
print("Embed   :", EMBEDDING_MODEL)

Project : https://aibslab
Chat    : gpt-4.1-mini
Embed   : text-embedding-3-large


!!! note "Expected output"
    ```
    Project : https://<account>.services.ai.azure.com/api/projects/<project>
    Chat    : gpt-4.1-mini
    Embed   : text-embedding-3-large
    ```
    The values come straight from your `.env` — no secrets in the notebook.

## 2. Build the client

`DefaultAzureCredential` uses your `az login` identity (or a managed identity in
production). `AIProjectClient` is constructed from the project endpoint + that
credential; `get_openai_client()` returns the OpenAI-compatible client wired to your
project.

In [3]:
from urllib.parse import urlparse
from azure.identity import AzureCliCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from openai import OpenAI

credential     = AzureCliCredential()  # faster + reliable than DefaultAzureCredential here
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# NOTE: the project /openai/v1 route serves chat & responses but 404s on embeddings
# for this project. The account-scoped Azure OpenAI endpoint serves all three, so we
# derive it from PROJECT_ENDPOINT and point the OpenAI client there.
_account = urlparse(PROJECT_ENDPOINT).hostname.split(".")[0]
ACCOUNT_OPENAI = f"https://{_account}.cognitiveservices.azure.com/openai/v1/"
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
openai_client  = OpenAI(base_url=ACCOUNT_OPENAI, api_key=token_provider())

print("project_client : ready")
# print only first 15 characters of account endpoint for security reasons
print("openai_client  : ready ->", ACCOUNT_OPENAI[:15])

project_client : ready
openai_client  : ready -> https://aibslab


!!! note "Expected output"
    ```
    project_client : ready
    openai_client  : ready
    ```
    A `DefaultAzureCredential` error here almost always means you need to run
    `az login`; a `403` means your identity lacks the **Azure AI Developer** role on
    the project.

## 3. Chat completions

The classic chat surface. You pass the **deployment name** (not a raw model id) and a
list of messages.

In [4]:
response = openai_client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise technical assistant."},
        {"role": "user",   "content": "What is catastrophic forgetting in neural networks?"},
    ],
)

print("Model  :", response.model)
print("Tokens :", response.usage.total_tokens)
print()
print(response.choices[0].message.content)

Model  : gpt-4.1-mini-2025-04-14
Tokens : 102

Catastrophic forgetting in neural networks refers to the phenomenon where a model, after being trained on new data or tasks, suddenly and significantly loses performance on previously learned tasks. This happens because the network's weights are updated to fit the new information, effectively overwriting or interfering with the representations learned for earlier tasks. It is a major challenge in continual learning and sequential task training.


!!! note "Expected output"
    ```
    Model  : gpt-4.1-mini
    Tokens : 142
    Catastrophic forgetting is the tendency of a neural network to abruptly lose
    knowledge of previously learned tasks when it is trained on a new task...
    ```
    Token counts and wording will vary; the shape is what matters.

## 4. Embeddings

Turn text into vectors — the foundation for retrieval. You'll lean on this in
[M4 · Grounding/RAG](../04-grounding-rag-foundry-iq/). One call
embeds a batch of strings.

In [5]:
texts = [
    "Microsoft Foundry centralises model governance behind one platform.",
    "Embeddings turn text into vectors for semantic search.",
    "Each project authenticates with DefaultAzureCredential.",
]

result = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)

print("Model      :", EMBEDDING_MODEL)
print("Dimensions :", len(result.data[0].embedding))
for i, item in enumerate(result.data):
    v = item.embedding
    print(f"[{i}] [{v[0]:.4f}, {v[1]:.4f}, {v[2]:.4f}, ...]  ({len(v)} dims)")

Model      : text-embedding-3-large
Dimensions : 3072
[0] [-0.0093, 0.0021, -0.0327, ...]  (3072 dims)
[1] [-0.0003, 0.0187, -0.0178, ...]  (3072 dims)
[2] [-0.0219, -0.0443, -0.0081, ...]  (3072 dims)


!!! note "Expected output"
    ```
    Model      : text-embedding-3-large
    Dimensions : 3072
    [0] [-0.0123, 0.0456, -0.0789, ...]  (3072 dims)
    [1] [0.0234, -0.0567, 0.0891, ...]  (3072 dims)
    [2] [-0.0345, 0.0678, -0.0912, ...]  (3072 dims)
    ```

## 5. Streaming

For responsive UIs, stream tokens as they're generated instead of waiting for the full
response.

In [6]:
stream = openai_client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is Microsoft Foundry?"}],
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

Microsoft

 Found

ry

 is

 a

...

 and

 co

-selling

 opportunities

.

!!! note "Expected output"
    The sentence prints **incrementally**, a few tokens at a time:
    ```
    Microsoft Foundry is Azure's unified platform-as-a-service for building, governing,
    and operating enterprise AI models, agents, and apps.
    ```

## 6. The Responses API

The **Responses API** is the modern, stateful surface that powers **agents** and
**tools** in every later lab. The minimal call takes a model and an `input`; the reply
is in `output_text`.

In [7]:
response = openai_client.responses.create(
    model=CHAT_MODEL,
    input="Name a planet with rings, in one short sentence.",
)

print(response.output_text)

Saturn is a planet with rings.


!!! note "Expected output"
    ```
    Saturn is a planet famous for its prominent ring system.
    ```

!!! tip "Why this matters"
    Hold onto this call. In the next lab you'll wrap a model in an **agent definition**
    and invoke it through this exact `responses.create(...)` surface — just with an
    `agent_reference` attached.

## 🧪 Your turn

1. **Swap the model.** If you deployed a reasoning model, set `REASONING_MODEL` in your
   `.env`, read it in cell 1, and re-run the Responses API call with it. Note how the
   answer style changes.
2. **Compare token usage.** Ask the chat model a long question vs. a short one and print
   `response.usage.total_tokens` for each.
3. **Embed and compare.** Embed two *similar* sentences and two *different* ones, then
   compute cosine similarity (`numpy.dot` on normalized vectors) — similar sentences
   should score higher.

---

✅ **You made chat, embedding, streaming, and Responses API calls from one client.**
Next: wrap a model in a versioned **agent** and invoke it.
→ **[M2 · Your First Agent](../02-your-first-agent/)**

## ✅ Your turn — solutions

The cells below complete the three **Your turn** challenges. They reuse the
`openai_client`, `CHAT_MODEL`, and `EMBEDDING_MODEL` created earlier in this
notebook, so run the notebook top-to-bottom first.


### 1 · Swap the model (reasoning model)

We read `REASONING_MODEL` from the environment and re-run the **Responses API**
call with it, alongside the base chat model, to compare answer style. Make sure
`REASONING_MODEL` is set in your `.env` (e.g. `REASONING_MODEL=o3`) and that the
deployment exists in your project.


In [8]:
import os

REASONING_MODEL = os.environ.get("REASONING_MODEL", "o3")
prompt = "Name a planet with rings, in one short sentence."

reasoning = openai_client.responses.create(model=REASONING_MODEL, input=prompt)
baseline  = openai_client.responses.create(model=CHAT_MODEL, input=prompt)

print(f"[{REASONING_MODEL:12}] {reasoning.output_text}")
print(f"[{CHAT_MODEL:12}] {baseline.output_text}")


[o3          ] Saturn is a planet with rings.
[gpt-4.1-mini] Saturn is a planet with rings.


### 2 · Compare token usage

Ask the chat model a **short** question and a **long** one, then print
`response.usage.total_tokens` for each. Longer prompts and longer answers cost
more tokens — the breakdown makes that concrete.


In [9]:
short_q = "What is a vector?"
long_q  = (
    "Explain, in detail, how transformer self-attention works, why it scales "
    "quadratically with sequence length, and what techniques mitigate that cost."
)

for label, question in [("short", short_q), ("long", long_q)]:
    r = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": question}],
    )
    u = r.usage
    print(f"{label:5} | prompt={u.prompt_tokens:5}  completion={u.completion_tokens:5}  total={u.total_tokens:5}")


short | prompt=   12  completion=  270  total=  282


long  | prompt=   35  completion= 1301  total= 1336


### 3 · Embed & compare (cosine similarity)

Embed two **similar** sentences and two **different** ones, then compute cosine
similarity with `numpy`. Semantically similar sentences should score noticeably
higher than unrelated ones.


In [10]:
import numpy as np

sentences = {
    "a1": "The cat sat quietly on the warm windowsill.",
    "a2": "A feline rested calmly by the sunny window.",     # similar to a1
    "b1": "Quarterly revenue exceeded analyst expectations.",
    "b2": "The rocket achieved orbit after a flawless launch.",  # different from b1
}

emb  = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=list(sentences.values()))
vecs = {k: np.array(d.embedding) for k, d in zip(sentences, emb.data)}

def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"similar   (a1, a2): {cosine(vecs['a1'], vecs['a2']):.4f}")
print(f"different (b1, b2): {cosine(vecs['b1'], vecs['b2']):.4f}")
print(f"cross     (a1, b1): {cosine(vecs['a1'], vecs['b1']):.4f}")

assert cosine(vecs['a1'], vecs['a2']) > cosine(vecs['a1'], vecs['b1']), "similar pair should score higher"
print("\n✔ Similar sentences scored higher than unrelated ones.")


similar   (a1, a2): 0.7354
different (b1, b2): 0.2132
cross     (a1, b1): 0.1211

✔ Similar sentences scored higher than unrelated ones.
